# Fig. 6B: ISG induction associated with Chaphamaparvovirus infection

Additional BLASTx top-hit tables used to refine negative controls.
Run `Fig6B_taxonkit.sh` at the indicated step, then finish this notebook before running `Fig6B.ipynb`.


In [1]:
import os
import pandas as pd
import numpy as np


In [2]:
# Define paths
basedir = "/path/to/project"

# Output directory
output_base = f"{basedir}/output/revise_final/Fig6B/blastx"

# Input files
blacklist_path = f"{basedir}/data/black_list_TN_ISG_low_new_format.txt"
batches = [
    {
        "name": name,
        "path_in": f"{basedir}/data/{name}_blastx_for_rna_seq/blastout_all_protein/",
        "path_out": f"{output_base}/{name}_blastx/",
    }
    for name in ("260209", "260302")
]

for batch in batches:
    os.makedirs(batch["path_out"], exist_ok=True)
    print("saving files:", batch["path_out"])


saving files: /Users/coco/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx/output/revise_final/Fig6B/blastx/260209_blastx/
saving files: /Users/coco/iLab/itolab_backup/backup-latest/Lab/projects/2507blastx/output/revise_final/Fig6B/blastx/260302_blastx/


In [3]:
def get_all_taxids(root_dir, takonkit_path):
    """Collect taxids from the selected BLASTx top hits in all samples."""
    taxids = set()
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            if not filename.endswith('.txt'):
                continue
            full_path = os.path.join(dirpath, filename)
            col_name = [
                "qseqid", "sseqid", "pident", "length",
                "mismatch", "gapopen", "qstart", "qend",
                "sstart", "send", "evalue", "bitscore",
                "qframe", "staxids", "stitle", "qlen",
                "slen",
            ]
            if os.path.getsize(full_path) == 0:
                print('file size 0 ')
                continue
            one_sample = pd.read_table(full_path, header=None, dtype={13: str})
            one_sample.columns = col_name
            if one_sample['bitscore'].isna().sum() > 0:
                print('NaN', filename)
            if one_sample['bitscore'].dtype != np.float64 and one_sample['bitscore'].dtype != int:
                print('dtype', filename, one_sample['bitscore'].dtype)
                print(one_sample['bitscore'])
            # Sort hits by bit score
            one_sample_sored = one_sample.sort_values('bitscore', ascending=False)
            # Remove hits without taxids
            one_sample_sored = one_sample_sored.dropna(subset=['staxids'])
            # Keep the top hit for each query
            one_sample_uniq = one_sample_sored.groupby('qseqid').first()
            for line_taxid in one_sample_uniq['staxids'].to_list():
                if line_taxid is None or line_taxid != line_taxid:
                    continue
                raw_taxids = line_taxid.split(';')
                for i in range(len(raw_taxids)):
                    taxids.add(raw_taxids[i])
    with open(takonkit_path, 'w') as f:
        for taxid in sorted(list(taxids)):
            f.write(taxid + '\n')
    print(f'{len(taxids)} taxids in total')
    return taxids

def get_viral_hits(input_file, out_dir, lineage_info, blacklst):
    """Annotate top hits and save viral hits after phage and blacklist filtering."""
    num_ambiguous_hit = 0
    output_folder = out_dir + '/taxonomy_output/'
    if not os.path.exists(output_folder):
        os.mkdir(output_folder)
    output_all_path = output_folder + '/' + input_file.split('/')[-1].split('.')[0] + '.blastx.taxonomy.txt'
    output_viral_path = output_folder + '/' + input_file.split('/')[-1].split('.')[0] + '.blastx.virus.txt'
    output_viral_black_path = output_folder + '/' + input_file.split('/')[-1].split('.')[0] + '.blastx.virus.black.txt'
    col_name = [
        "qseqid", "sseqid", "pident", "length",
        "mismatch", "gapopen", "qstart", "qend",
        "sstart", "send", "evalue", "bitscore",
        "qframe", "staxids", "stitle", "qlen",
        "slen",
    ]
    one_sample = pd.read_table(input_file, header=None, dtype={13: str})
    one_sample.columns = col_name
    # Sort hits by bit score
    one_sample_sored = one_sample.sort_values('bitscore', ascending=False)
    # Remove hits without taxids
    one_sample_sored = one_sample_sored.dropna(subset=['staxids'])
    # Keep the top hit for each query
    one_sample_uniq = one_sample_sored.groupby('qseqid').first()
    num_all_hits = len(set(one_sample_uniq.index.to_list()))
    # Add taxonomic annotations
    one_sample_tax = one_sample_uniq.copy()
    one_sample_tax['staxids_lst'] = one_sample_tax['staxids'].apply(lambda x: x.split(';') if x is not None and x == x else x)
    one_sample_tax['name_lst'] = one_sample_tax['staxids_lst'].apply(lambda x: [lineage_info.loc[taxid, 'fgs'] for taxid in x] if x is not None and x == x else x)
    one_sample_tax['group_lst'] = one_sample_tax['staxids_lst'].apply(lambda x: [lineage_info.loc[taxid, 'domain'] for taxid in x] if x is not None and x == x else x)
    for idx in one_sample_tax.index:
        groups = one_sample_tax.loc[idx, 'group_lst']
        if (groups is not None and groups == groups) and any(('Viruses' in item for item in groups)) and (len(set(groups)) > 1):
            num_ambiguous_hit += 1
    one_sample_tax['taxid_blastx'] = one_sample_tax['staxids_lst'].apply(lambda x: x[0] if x is not None and x == x else x)
    one_sample_tax['family'] = one_sample_tax['staxids_lst'].apply(lambda x: lineage_info.loc[x[0], 'family'] if x is not None and x == x else x)
    one_sample_tax['genus'] = one_sample_tax['staxids_lst'].apply(lambda x: lineage_info.loc[x[0], 'genus'] if x is not None and x == x else x)
    one_sample_tax['species'] = one_sample_tax['staxids_lst'].apply(lambda x: lineage_info.loc[x[0], 'species'] if x is not None and x == x else x)
    # Retain hits only when all taxonomic groups are viral
    one_sample_tax['is_virus'] = one_sample_tax['group_lst'].apply(lambda x: 1 if (x is not None and x == x) and all(('Viruses' in item for item in x)) else 0)
    one_sample_virus = one_sample_tax.copy()
    one_sample_virus = one_sample_virus[one_sample_virus['is_virus'] == 1]
    num_viral_hits = len(set(one_sample_virus.index.to_list()))
    blackvirus = []
    with open(blacklst, 'r') as f_in:
        for line in f_in:
            blackvirus.append(line.strip())
    # Exclude blacklisted viral families
    one_sample_black_removed = one_sample_virus.copy()
    one_sample_black_removed = one_sample_black_removed[~one_sample_black_removed['family'].isin(blackvirus)]
    num_viral_hits_after_filter = len(set(one_sample_black_removed.index.to_list()))
    # Save per-sample annotation tables
    one_sample_tax.to_csv(output_all_path, sep='\t')
    one_sample_virus.to_csv(output_viral_path, sep='\t')
    one_sample_black_removed.to_csv(output_viral_black_path, sep='\t')
    return (one_sample_black_removed, num_all_hits, num_viral_hits, num_viral_hits_after_filter, num_ambiguous_hit)


# TaxonKit


In [4]:
for batch in batches:
    print("Batch:", batch["name"])
    taxids = get_all_taxids(batch["path_in"], batch["path_out"] + "taxids.txt")


Batch: 260209
1956 taxids in total
Batch: 260302
383 taxids in total


Run `Fig6B_taxonkit.sh` in the taxonkit environment.


# Identify phages


In [5]:
def create_phage_tables(path_out):
    """Identify phage taxa and save the phage annotation table and taxid list."""
    # Read TaxonKit output
    taxinfo = pd.read_table(path_out + 'taxids_long.txt', header=None, dtype={0: str})
    taxinfo.columns = ['taxid', 'lineage']
    # Split the lineage into taxonomic ranks
    taxinfo['domain'] = taxinfo['lineage'].apply(lambda x: x.split(';')[0] if x == x else x)
    taxinfo['kingdom'] = taxinfo['lineage'].apply(lambda x: x.split(';')[1] if x == x else x)
    taxinfo['phylum'] = taxinfo['lineage'].apply(lambda x: x.split(';')[2] if x == x else x)
    taxinfo['class'] = taxinfo['lineage'].apply(lambda x: x.split(';')[3] if x == x else x)
    taxinfo['order'] = taxinfo['lineage'].apply(lambda x: x.split(';')[4] if x == x else x)
    taxinfo['family'] = taxinfo['lineage'].apply(lambda x: x.split(';')[5] if x == x else x)
    taxinfo['genus'] = taxinfo['lineage'].apply(lambda x: x.split(';')[6] if x == x else x)
    taxinfo['species'] = taxinfo['lineage'].apply(lambda x: x.split(';')[7] if x == x else x)
    print('#NaN:', taxinfo.isna().sum().sum())
    taxinfo = taxinfo.fillna('-')
    print('Original size:', taxinfo.shape[0])
    # Identify phage taxa
    taxinfo['is_phage'] = 0
    taxinfo['r1_Caudoviricetes'] = 0
    taxinfo['r2_includes_phage'] = 0
    taxinfo['r3_Inoviridae'] = 0
    taxinfo['r4_Lavidaviridae'] = 0
    taxinfo['r5_Microviridae'] = 0
    phage_group_order = set()
    for idx in taxinfo.index:
        # 1. Caudoviricetes
        if taxinfo.loc[idx, 'class'] == 'Caudoviricetes':
            taxinfo.loc[idx, 'is_phage'] = 1
            taxinfo.loc[idx, 'r1_Caudoviricetes'] = 1
        # 2. Unclassified viral species containing "phage"
        if 'phage' in taxinfo.loc[idx, 'species'] and taxinfo.loc[idx, 'domain'] == 'unclassified Viruses domain':
            taxinfo.loc[idx, 'is_phage'] = 1
            taxinfo.loc[idx, 'r2_includes_phage'] = 1
            phage_group_order.add(taxinfo.loc[idx, 'order'])
        # 3. Inoviridae
        if taxinfo.loc[idx, 'family'] == 'Inoviridae':
            taxinfo.loc[idx, 'is_phage'] = 1
            taxinfo.loc[idx, 'r3_Inoviridae'] = 1
        # 4. Lavidaviridae
        if taxinfo.loc[idx, 'family'] == 'Lavidaviridae':
            taxinfo.loc[idx, 'is_phage'] = 1
            taxinfo.loc[idx, 'r4_Lavidaviridae'] = 1
        # 5. Microviridae
        if taxinfo.loc[idx, 'family'] == 'Microviridae':
            taxinfo.loc[idx, 'is_phage'] = 1
            taxinfo.loc[idx, 'r5_Microviridae'] = 1
    phage_group_order = sorted(list(phage_group_order))
    print('phage order:', phage_group_order)
    print('Caudoviricetes:', taxinfo.loc[taxinfo['r1_Caudoviricetes'] == 1, :].shape[0])
    print('phage:', taxinfo.loc[taxinfo['r2_includes_phage'] == 1, :].shape[0])
    print(' * not Caudoviricetes:', taxinfo.loc[(taxinfo['r1_Caudoviricetes'] == 0) & (taxinfo['r2_includes_phage'] == 1), :].shape[0])
    print('Inoviridae:', taxinfo.loc[taxinfo['r3_Inoviridae'] == 1, :].shape[0])
    print('Lavidaviridae:', taxinfo.loc[taxinfo['r4_Lavidaviridae'] == 1, :].shape[0])
    print('Microviridae:', taxinfo.loc[taxinfo['r5_Microviridae'] == 1, :].shape[0])
    # Save the phage annotation table and taxid list
    taxinfo.to_csv(path_out + 'taxids_phage.tsv', sep='\t')
    phages = taxinfo.loc[taxinfo['is_phage'] == 1, 'taxid'].to_list()
    with open(path_out + 'phage_list.txt', 'w') as f:
        for pid in phages:
            f.write(pid + '\n')


In [6]:
for batch in batches:
    print("Batch:", batch["name"])
    create_phage_tables(batch["path_out"])


Batch: 260209
#NaN: 0
Original size: 1956
phage order: ['unclassified Caudoviricetes order', 'unclassified Viruses order']
Caudoviricetes: 103
phage: 47
 * not Caudoviricetes: 1
Inoviridae: 0
Lavidaviridae: 0
Microviridae: 0
Batch: 260302
#NaN: 0
Original size: 383
phage order: []
Caudoviricetes: 0
phage: 0
 * not Caudoviricetes: 0
Inoviridae: 0
Lavidaviridae: 0
Microviridae: 0


# Add taxonomy


In [7]:
def read_lineage_info(path_out):
    """Read taxonomic lineages and label phage taxa as phage_group."""
    phage_path = path_out + 'phage_list.txt'
    taxid_in = path_out + 'taxids_lineage.txt'
    # Read the lineage table
    lineage_info = pd.read_table(taxid_in, header=None)
    lineage_info.columns = ['taxid', 'domain', 'fgs']
    print('#NaN beginning:', lineage_info.isna().sum().sum())
    nan_rows = lineage_info[lineage_info.isna().any(axis=1)]
    print(nan_rows)
    print()
    # Add family, genus and species columns
    lineage_info['taxid'] = lineage_info['taxid'].astype(str)
    lineage_info = lineage_info.set_index('taxid')
    lineage_info['family'] = lineage_info['fgs'].apply(lambda x: x.split(';')[0] if x == x else np.nan)
    lineage_info['genus'] = lineage_info['fgs'].apply(lambda x: x.split(';')[1] if x == x else np.nan)
    lineage_info['species'] = lineage_info['fgs'].apply(lambda x: x.split(';')[2] if x == x else np.nan)
    # Label phage taxa as phage_group
    phage_ids = []
    with open(phage_path, 'r') as f:
        for line in f:
            line = line.strip()
            phage_ids.append(line)
    for idx in lineage_info.index:
        if idx in phage_ids:
            lineage_info.loc[idx, 'domain'] = 'phage_group'
    print('#NaN end:', lineage_info.isna().sum().sum())
    nan_rows = lineage_info[lineage_info.isna().any(axis=1)]
    print(nan_rows)
    print()
    lineage_info = lineage_info.fillna('-')
    print(f'{lineage_info.shape[0]} taxids in total\n')
    print('All domain:', set(lineage_info['domain'].to_list()), '\n')
    for domain in set(lineage_info['domain'].to_list()):
        print(f'##### {domain} ####')
        print(f"size: {lineage_info[lineage_info['domain'] == domain].shape[0]}")
        print(lineage_info[['domain', 'species']][lineage_info['domain'] == domain].head(3))
        print()
    return lineage_info


In [8]:
lineage_by_batch = {}
for batch in batches:
    print("Batch:", batch["name"])
    lineage_by_batch[batch["name"]] = read_lineage_info(batch["path_out"])


Batch: 260209
#NaN beginning: 0
Empty DataFrame
Columns: [taxid, domain, fgs]
Index: []

#NaN end: 0
Empty DataFrame
Columns: [domain, fgs, family, genus, species]
Index: []

1956 taxids in total

All domain: {'Eukaryota', 'unclassified other entries domain', 'unclassified Viruses domain', 'Bacteria', 'phage_group', 'unclassified unclassified entries domain'} 

##### Eukaryota ####
size: 1089
            domain              species
taxid                                  
10020    Eukaryota      Dipodomys ordii
1002788  Eukaryota  Indicator indicator
10029    Eukaryota   Cricetulus griseus

##### unclassified other entries domain ####
size: 12
                                    domain  \
taxid                                        
1521026  unclassified other entries domain   
1547346  unclassified other entries domain   
1565116  unclassified other entries domain   

                                                   species  
taxid                                                    

# Extract viral top hits


In [9]:
def count_viral_contigs(path_in: str, infection_th=1):
    """Collect filtered viral hits across all RNA-seq samples."""
    num_amb_hit = 0
    is_fist_file = 1
    # Read all BLAST result files
    for dirpath, dirnames, filenames in os.walk(path_in):
        for filename in filenames:
            if not filename.endswith('.txt'):
                continue
            full_path = os.path.join(dirpath, filename)
            if os.path.getsize(full_path) == 0:
                print('filesize 0')
                continue
            df_one_sample, num_all_contigs, num_viral_contigs, num_viral_black, num_amb_hit_each = get_viral_hits(full_path, path_out, lineage_info, blacklist_path)
            num_amb_hit += num_amb_hit_each
            if is_fist_file == 1:
                df_all = df_one_sample.copy()
                is_fist_file = 0
            else:
                df_all = pd.concat([df_all, df_one_sample])
    print('Virus ambiguous hit:', num_amb_hit)
    print('=======')
    print('=======')
    return df_all


# Save tables


In [10]:
for batch in batches:
    path_in = batch["path_in"]
    path_out = batch["path_out"]
    lineage_info = lineage_by_batch[batch["name"]]
    print("Batch:", batch["name"])
    df_all = count_viral_contigs(path_in, 1)
    df = df_all.copy().reset_index()
    print('shape:', df.shape)
    # Derive sample identifiers from query names
    df['sample'] = df['qseqid'].apply(lambda x: x.split('_')[0])
    # Add viral-contig counts per sample
    virus_hit_count = pd.DataFrame(df.value_counts('sample'))
    df = df.merge(virus_hit_count, on='sample', how='left')
    # Sort by sample, then by viral contig count
    df = df.sort_values('sample')
    df = df.sort_values('count', ascending=False)
    print('#NaN:', df.isna().sum().sum())
    # Save the viral contig table
    df.to_csv(path_out + 'virus_all.txt', sep='\t')
    df.head(5)


Batch: 260209
Virus ambiguous hit: 29
shape: (913, 25)
#NaN: 0
Batch: 260302
Virus ambiguous hit: 0
shape: (6, 25)
#NaN: 0
